# Chia frame ảnh

In [1]:
import cv2
import os
path = r"D:\SYLLABUS\AIP491\demo\test_processing"
cap = cv2.VideoCapture(r"D:\SYLLABUS\AIP491\data\cam-03062025_635_705\NO20250603-184943-009947F.MP4")
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if frame_count % 30 == 0: 
        image_path = os.path.join(path, f"frame_{frame_count}.jpg")
        cv2.imwrite(image_path, frame)
    frame_count += 1
cap.release()

# Paper

## Vẽ Contour

In [30]:
import cv2
import numpy as np

# Đọc ảnh
img_path = r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg"
img = cv2.imread(img_path)
b, g, r = cv2.split(img)


def filter_region_mask(region_mask, min_area=100):
    contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) >= min_area]
    
    new_mask = np.zeros_like(region_mask)
    cv2.drawContours(new_mask, large_contours, -1, 255, -1)  # Fill vùng hợp lệ
    return new_mask

# ----------------- REGION 1: Core Glare -----------------
# region1_mask = ((b == 255) & (g == 255) & (r == 255)).astype(np.uint8) * 255
region1_mask = (
    (b >= 250) & (b <= 255) &
    (g >= 250) & (g <= 255) &
    (r >= 250) & (r <= 255)
).astype(np.uint8) * 255

# ----------------- REGION 2: Strong Glare -----------------
cond_b_2 = (b >= 200)
cond_g_2 = (g >= 200)
cond_r_2 = (r >= 200)
sum_2 = cond_b_2.astype(np.uint8) + cond_g_2.astype(np.uint8) + cond_r_2.astype(np.uint8)
region2_mask = ((sum_2 >= 2) & (region1_mask == 0)).astype(np.uint8) * 255

# ----------------- REGION 3: Mild Glare -----------------
cond_b_3 = (b >= 150) & (b < 200)
cond_g_3 = (g >= 150) & (g < 200)
cond_r_3 = (r >= 150) & (r < 200)
sum_3 = cond_b_3.astype(np.uint8) + cond_g_3.astype(np.uint8) + cond_r_3.astype(np.uint8)
region3_mask = ((sum_3 >= 2) & (region1_mask == 0) & (region2_mask == 0)).astype(np.uint8) * 255

# ----------------- TÌM CONTOURS -----------------
contours1, _ = cv2.findContours(region1_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2, _ = cv2.findContours(region2_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours3, _ = cv2.findContours(region3_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# ----------------- VẼ CONTOURS TRÊN ẢNH -----------------
region1_mask = filter_region_mask(region1_mask, min_area=100)
region2_mask = filter_region_mask(region2_mask, min_area=100)
region3_mask = filter_region_mask(region3_mask, min_area=100)
output_contour = img.copy()

cv2.drawContours(output_contour, contours1, -1, (0, 0, 255), 1)   # Vùng 1: đỏ
cv2.drawContours(output_contour, contours2, -1, (0, 255, 255), 1) # Vùng 2: vàng
cv2.drawContours(output_contour, contours3, -1, (255, 0, 0), 1)   # Vùng 3: blue

# ----------------- HIỂN THỊ -----------------
cv2.imshow("Contours on Glare Regions", output_contour)
# cv2.imshow("Region 1 (Mask)", region1_mask)
# cv2.imshow("Region 2 (Mask)", region2_mask)
# cv2.imshow("Region 3 (Mask)", region3_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()


# Xử lý khu vực 1

In [31]:
import cv2
import numpy as np

# ----------- Đọc ảnh và tách kênh -----------
img_path = r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg"
img = cv2.imread(img_path)
b, g, r = cv2.split(img)

def shrink_region1_mask(region_mask, shrink_ratio=0.5, min_area=100):
    contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    new_mask = np.zeros_like(region_mask)
    for cnt in contours:
        if cv2.contourArea(cnt) >= min_area:
            (x, y), radius = cv2.minEnclosingCircle(cnt)
            center = (int(x), int(y))
            radius = int(radius * shrink_ratio)
            cv2.circle(new_mask, center, radius, 255, -1)
    return new_mask


# ----------- Tạo mặt nạ cho từng vùng -----------

# Region 1: Core Glare (cả 3 kênh = 255)
# region1_mask = ((b == 255) & (g == 255) & (r == 255)).astype(np.uint8) * 255
region1_mask = (
    (b >= 250) & (b <= 255) &
    (g >= 250) & (g <= 255) &
    (r >= 250) & (r <= 255)
).astype(np.uint8) * 255

# Region 2: Ít nhất 2 kênh trong khoảng 200–255, loại trừ Region 1
cond_b_2 = (b >= 200)
cond_g_2 = (g >= 200)
cond_r_2 = (r >= 200)
sum_2 = cond_b_2.astype(np.uint8) + cond_g_2.astype(np.uint8) + cond_r_2.astype(np.uint8)
region2_mask = ((sum_2 >= 2) & (region1_mask == 0)).astype(np.uint8) * 255

# Region 3: Ít nhất 2 kênh trong khoảng 150–200, loại trừ Region 1 & 2
cond_b_3 = (b >= 150) & (b < 200)
cond_g_3 = (g >= 150) & (g < 200)
cond_r_3 = (r >= 150) & (r < 200)
sum_3 = cond_b_3.astype(np.uint8) + cond_g_3.astype(np.uint8) + cond_r_3.astype(np.uint8)
region3_mask = ((sum_3 >= 2) & (region1_mask == 0) & (region2_mask == 0)).astype(np.uint8) * 255

# ----------- Tìm contour cho từng vùng -----------
contours1, _ = cv2.findContours(region1_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2, _ = cv2.findContours(region2_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours3, _ = cv2.findContours(region3_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# ----------- Tạo bản sao ảnh để xử lý vùng 1 -----------
processed_img = img.copy()

# ----------- Xử lý pixel vùng 1: giảm sáng nếu >150 -----------
mask1_bool = region1_mask == 255
for c in range(3):  # 0: B, 1: G, 2: R
    channel = processed_img[:, :, c]
    channel_mask = (channel > 150) & mask1_bool
    channel[channel_mask] = np.maximum(0, channel[channel_mask] - 80)

region1_mask = shrink_region1_mask(region1_mask, shrink_ratio=0.5, min_area=100)

# ----------- Vẽ contour lên ảnh đã xử lý -----------
output_with_contours = processed_img.copy()
cv2.drawContours(output_with_contours, contours1, -1, (0, 0, 255), 1)   # Red - Region 1
cv2.drawContours(output_with_contours, contours2, -1, (0, 255, 255), 1) # Yellow - Region 2
cv2.drawContours(output_with_contours, contours3, -1, (255, 0, 0), 1)   # Blue - Region 3



# ----------- Hiển thị kết quả -----------
cv2.imshow("Original Image", img)
cv2.imshow("Processed Image (Region 1 Reduced)", processed_img)
# cv2.imshow("Contours on Processed Image", output_with_contours)
# cv2.imshow("Region 1 Mask", region1_mask)
# cv2.imshow("Region 2 Mask", region2_mask)
# cv2.imshow("Region 3 Mask", region3_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()


# Xử lý khu vực 2

In [8]:
import cv2
import numpy as np

# ----------- Đọc ảnh -----------
img_path = r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg"
img = cv2.imread(img_path)
b, g, r = cv2.split(img)

# ----------- Tạo mặt nạ cho từng vùng -----------

# Region 1: Core Glare (cả 3 kênh = 255)
region1_mask = ((b == 255) & (g == 255) & (r == 255)).astype(np.uint8) * 255

# Region 2: ≥2 kênh từ 200–255, loại trừ Region 1
cond_b_2 = (b >= 200)
cond_g_2 = (g >= 200)
cond_r_2 = (r >= 200)
sum_2 = cond_b_2.astype(np.uint8) + cond_g_2.astype(np.uint8) + cond_r_2.astype(np.uint8)
region2_mask = ((sum_2 >= 2) & (region1_mask == 0)).astype(np.uint8) * 255

# Region 3: ≥2 kênh từ 150–200, loại trừ Region 1 & 2
cond_b_3 = (b >= 150) & (b < 200)
cond_g_3 = (g >= 150) & (g < 200)
cond_r_3 = (r >= 150) & (r < 200)
sum_3 = cond_b_3.astype(np.uint8) + cond_g_3.astype(np.uint8) + cond_r_3.astype(np.uint8)
region3_mask = ((sum_3 >= 2) & (region1_mask == 0) & (region2_mask == 0)).astype(np.uint8) * 255

# ----------- Xử lý Region 1: giảm sáng nếu >150 -----------
processed_img = img.copy()
mask1_bool = region1_mask == 255

for c in range(3):  # B, G, R
    channel = processed_img[:, :, c]
    channel_mask = (channel > 150) & mask1_bool
    channel[channel_mask] = np.maximum(0, channel[channel_mask] - 80)

# ----------- Xử lý Region 2: dùng LUT theo gamma -----------
region2_coords = np.where(region2_mask == 255)
intensity_vals = img[region2_coords]
gray_vals = intensity_vals.mean(axis=1)  # cường độ xám trung bình
max_intensity = int(np.max(gray_vals))
gamma = 0.6  # <1 làm sáng, >1 làm tối

# Tạo LUT theo công thức gamma
LUT = np.array([
    np.clip(max_intensity * ((i / max_intensity) ** gamma), 0, 255)
    for i in range(0, max_intensity + 1)
], dtype=np.uint8)

# Ánh xạ LUT cho từng kênh
processed_img2 = processed_img.copy()
for c in range(3):  # B, G, R
    channel = processed_img2[:, :, c]
    region_values = channel[region2_coords]
    region_values_clipped = np.clip(region_values, 0, max_intensity)
    channel[region2_coords] = LUT[region_values_clipped]

# ----------- Vẽ contour trên ảnh đã xử lý -----------
contours1, _ = cv2.findContours(region1_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2, _ = cv2.findContours(region2_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours3, _ = cv2.findContours(region3_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

output_with_contours = processed_img2.copy()
cv2.drawContours(output_with_contours, contours1, -1, (0, 0, 255), 2)   # Red
cv2.drawContours(output_with_contours, contours2, -1, (0, 255, 255), 2) # Yellow
cv2.drawContours(output_with_contours, contours3, -1, (255, 0, 0), 2)   # Blue

# ----------- Hiển thị kết quả -----------
cv2.imshow("Original Image", img)
cv2.imshow("Processed Image (Region 1 + Region 2)", processed_img2)
cv2.imshow("Contours on Processed Image", output_with_contours)
# cv2.imshow("Region 1 Mask", region1_mask)
# cv2.imshow("Region 2 Mask", region2_mask)
# cv2.imshow("Region 3 Mask", region3_mask)
cv2.waitKey(0)
cv2.destroyAllWindows()


# Khu vực 3

In [10]:
import cv2
import numpy as np

# ----------- Đọc ảnh và tách kênh -----------
img_path = r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg"
img = cv2.imread(img_path)
b, g, r = cv2.split(img)

# ----------- Phát hiện 3 vùng sáng -----------

# Region 1: B = G = R = 255
region1_mask = ((b == 255) & (g == 255) & (r == 255)).astype(np.uint8) * 255

# Region 2: ≥ 2 kênh >= 200, không thuộc Region 1
cond_b2 = b >= 200
cond_g2 = g >= 200
cond_r2 = r >= 200
sum_2 = cond_b2.astype(np.uint8) + cond_g2.astype(np.uint8) + cond_r2.astype(np.uint8)
region2_mask = ((sum_2 >= 2) & (region1_mask == 0)).astype(np.uint8) * 255

# Region 3: ≥ 2 kênh từ 150–200, không thuộc Region 1 và 2
cond_b3 = (b >= 150) & (b < 200)
cond_g3 = (g >= 150) & (g < 200)
cond_r3 = (r >= 150) & (r < 200)
sum_3 = cond_b3.astype(np.uint8) + cond_g3.astype(np.uint8) + cond_r3.astype(np.uint8)
region3_mask = ((sum_3 >= 2) & (region1_mask == 0) & (region2_mask == 0)).astype(np.uint8) * 255

# ----------- Region 1: giảm sáng thủ công -----------
processed_img = img.copy()
mask1_bool = region1_mask == 255
for c in range(3):  # B, G, R
    channel = processed_img[:, :, c]
    channel_mask = (channel > 150) & mask1_bool
    channel[channel_mask] = np.maximum(0, channel[channel_mask] - 80)

# ----------- Region 2: Gamma LUT -----------
region2_coords = np.where(region2_mask == 255)
region2_vals = img[region2_coords]
gray_vals = region2_vals.mean(axis=1)
max_intensity = int(np.max(gray_vals))
gamma = 0.6
LUT = np.array([
    np.clip(max_intensity * ((i / max_intensity) ** gamma), 0, 255)
    for i in range(0, max_intensity + 1)
], dtype=np.uint8)

processed_img2 = processed_img.copy()
for c in range(3):  # B, G, R
    channel = processed_img2[:, :, c]
    region_values = channel[region2_coords]
    region_values_clipped = np.clip(region_values, 0, max_intensity)
    channel[region2_coords] = LUT[region_values_clipped]

# ----------- Region 3: Làm mềm bằng giá trị môi trường lân cận -----------
# Làm mờ ảnh toàn cục để đại diện ánh sáng lân cận
blurred_img_env = cv2.blur(processed_img2, (15, 15))

mask3_bool = region3_mask == 255
for c in range(3):
    channel = processed_img2[:, :, c]
    blurred_channel = blurred_img_env[:, :, c]
    channel[mask3_bool] = blurred_channel[mask3_bool]

# ----------- Vẽ contour từng vùng -----------
contours1, _ = cv2.findContours(region1_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2, _ = cv2.findContours(region2_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours3, _ = cv2.findContours(region3_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

final_output = processed_img2.copy()
cv2.drawContours(final_output, contours1, -1, (0, 0, 255), 2)   # Red
cv2.drawContours(final_output, contours2, -1, (0, 255, 255), 2) # Yellow
cv2.drawContours(final_output, contours3, -1, (255, 0, 0), 2)   # Blue

# ----------- Hiển thị kết quả -----------
cv2.imshow("Original Image", img)
# cv2.imshow("Region 1 Mask", region1_mask)
# cv2.imshow("Region 2 Mask", region2_mask)
# cv2.imshow("Region 3 Mask", region3_mask)
cv2.imshow("Final Processed Image with Contours", final_output)
cv2.waitKey(0)
cv2.destroyAllWindows()


# Hiển thị không có contour


In [11]:
import cv2
import numpy as np

# ----------- Đọc ảnh và tách kênh -----------
img_path = r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg"
img = cv2.imread(img_path)
b, g, r = cv2.split(img)

# ----------- Phát hiện 3 vùng sáng -----------

# Region 1: B = G = R = 255
region1_mask = ((b == 255) & (g == 255) & (r == 255)).astype(np.uint8) * 255

# Region 2: ≥ 2 kênh >= 200, không thuộc Region 1
cond_b2 = b >= 200
cond_g2 = g >= 200
cond_r2 = r >= 200
sum_2 = cond_b2.astype(np.uint8) + cond_g2.astype(np.uint8) + cond_r2.astype(np.uint8)
region2_mask = ((sum_2 >= 2) & (region1_mask == 0)).astype(np.uint8) * 255

# Region 3: ≥ 2 kênh từ 150–200, không thuộc Region 1 và 2
cond_b3 = (b >= 150) & (b < 200)
cond_g3 = (g >= 150) & (g < 200)
cond_r3 = (r >= 150) & (r < 200)
sum_3 = cond_b3.astype(np.uint8) + cond_g3.astype(np.uint8) + cond_r3.astype(np.uint8)
region3_mask = ((sum_3 >= 2) & (region1_mask == 0) & (region2_mask == 0)).astype(np.uint8) * 255

# ----------- Region 1: giảm sáng thủ công -----------
processed_img = img.copy()
mask1_bool = region1_mask == 255
for c in range(3):  # B, G, R
    channel = processed_img[:, :, c]
    channel_mask = (channel > 150) & mask1_bool
    channel[channel_mask] = np.maximum(0, channel[channel_mask] - 80)

# ----------- Region 2: Gamma LUT -----------
region2_coords = np.where(region2_mask == 255)
region2_vals = img[region2_coords]
gray_vals = region2_vals.mean(axis=1)
max_intensity = int(np.max(gray_vals))
gamma = 0.6
LUT = np.array([
    np.clip(max_intensity * ((i / max_intensity) ** gamma), 0, 255)
    for i in range(0, max_intensity + 1)
], dtype=np.uint8)

processed_img2 = processed_img.copy()
for c in range(3):  # B, G, R
    channel = processed_img2[:, :, c]
    region_values = channel[region2_coords]
    region_values_clipped = np.clip(region_values, 0, max_intensity)
    channel[region2_coords] = LUT[region_values_clipped]

# ----------- Region 3: Làm mềm bằng giá trị môi trường lân cận -----------
# Làm mờ ảnh toàn cục để đại diện ánh sáng lân cận
blurred_img_env = cv2.blur(processed_img2, (15, 15))

mask3_bool = region3_mask == 255
for c in range(3):
    channel = processed_img2[:, :, c]
    blurred_channel = blurred_img_env[:, :, c]
    channel[mask3_bool] = blurred_channel[mask3_bool]

# ----------- Hiển thị kết quả (không vẽ contour) -----------
cv2.imshow("Original Image", img)
cv2.imshow("Processed Image (All Regions)", processed_img2)
cv2.waitKey(0)
cv2.destroyAllWindows()


# Dùng thêm helper 

In [34]:
# Real-time automotive glare reduction pipeline (theo paper goc)

import cv2
import numpy as np

# --------------------------- REGION 1: MASKING ---------------------------
def process_region1(img, region1_mask):
    result = img.copy()
    for c in range(3):
        channel = result[:, :, c]
        mask = (channel > 150) & (region1_mask == 255)
        channel[mask] = np.maximum(0, channel[mask] - 50)

    # Shrink Region 1 mask by 50%
    contours, _ = cv2.findContours(region1_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    new_mask = np.zeros_like(region1_mask)
    for cnt in contours:
        (x, y), radius = cv2.minEnclosingCircle(cnt)
        center = (int(x), int(y))
        radius = int(radius * 0.5)
        cv2.circle(new_mask, center, radius, 255, -1)
    return result, new_mask

# --------------------------- REGION 2: GAMMA CORRECTION ---------------------------
def gamma_lut_correction(img, region2_mask, gamma=0.5):
    result = img.copy()
    if np.count_nonzero(region2_mask) == 0:
        return result

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    region2_pixels = gray[region2_mask == 255]
    max_val = np.max(region2_pixels) if region2_pixels.size > 0 else 255

    LUT = np.array([np.clip(max_val * ((i / max_val) ** gamma), 0, 255) for i in range(256)]).astype('uint8')

    for c in range(3):
        channel = result[:, :, c]
        channel[region2_mask == 255] = LUT[channel[region2_mask == 255]]
    return result

# --------------------------- REGION 3: PATCH-BASED ENHANCEMENT ---------------------------
def process_region3(img, region3_mask):
    result = img.copy()
    I = img.astype(np.float32)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    A = np.max(gray.astype(np.float32))

    transmission = np.ones_like(gray, dtype=np.float32)

    # Estimate transmission map t(p) per patch
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            if region3_mask[y, x] != 255:
                continue
            intensity = gray[y, x]
            if intensity <= 85:
                k = 3
            elif intensity <= 170:
                k = 5
            else:
                k = 9
            r = k // 2
            y1, y2 = max(0, y - r), min(img.shape[0], y + r + 1)
            x1, x2 = max(0, x - r), min(img.shape[1], x + r + 1)
            patch = gray[y1:y2, x1:x2].astype(np.float32)
            Ih = np.max(patch)
            t = (Ih - A) / (255.0 - A)
            t = max(t, 0.15)
            for c in range(3):
                result[y, x, c] = ((I[y, x, c] - A) / t) + A
    return np.clip(result, 0, 255).astype(np.uint8)

# --------------------------- FINAL SMOOTHING ---------------------------
def smooth_boundaries(processed_img, region_mask):
    blurred = cv2.GaussianBlur(processed_img, (5, 5), 0)
    return np.where(region_mask[..., None] == 255, blurred, processed_img)

# --------------------------- MAIN PIPELINE ---------------------------
def process_image_pipeline(img):
    b, g, r = cv2.split(img)
    region1_mask = ((b >= 240) & (g >= 240) & (r >= 240)).astype(np.uint8) * 255
    region2_mask = (((b >= 200) + (g >= 200) + (r >= 200)) >= 2).astype(np.uint8) * 255
    region2_mask = cv2.bitwise_and(region2_mask, cv2.bitwise_not(region1_mask))
    region3_mask = (((b >= 150) + (g >= 150) + (r >= 150)) >= 2).astype(np.uint8) * 255
    region3_mask = cv2.bitwise_and(region3_mask, cv2.bitwise_not(cv2.bitwise_or(region1_mask, region2_mask)))

    # Region 1
    img_r1, shrunk_mask1 = process_region1(img, region1_mask)

    # Region 2
    img_r2 = gamma_lut_correction(img, region2_mask)

    # Region 3
    img_r3 = process_region3(img, region3_mask)

    # Combine
    combined = img.copy()
    combined[shrunk_mask1 == 255] = img_r1[shrunk_mask1 == 255]
    combined[region2_mask == 255] = img_r2[region2_mask == 255]
    combined[region3_mask == 255] = img_r3[region3_mask == 255]

    # Smooth boundaries
    final = smooth_boundaries(combined, shrunk_mask1 | region2_mask | region3_mask)
    return final

# --------------------------- DEMO ---------------------------
if __name__ == "__main__":
    img = cv2.imread(r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg")
    result = process_image_pipeline(img)
    cv2.imshow("Glare Reduced", result)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

  


# Tối trong sáng ngoài

In [ ]:
import cv2 as cv 
import numpy as np
alpha = 1.0 # Simple contrast control
beta = 0    # Simple brightness control
img_path = r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_780_resize.jpg"
img = cv2.imread(img_path)
# b, g, r = cv2.split(img)

print(' Basic Linear Transforms ')
print('-------------------------')
try:
    alpha = float(input('* Enter the alpha value [1.0-3.0]: '))
    beta = int(input('* Enter the beta value [0-100]: '))
except ValueError:
    print('Error, not a number')

# Do the operation new_image(i,j) = alpha*image(i,j) + beta
# Instead of these 'for' loops we could have used simply:
new_image = cv.convertScaleAbs(img, alpha=alpha, beta=beta)
# but we wanted to show you how to access the pixels :)
b, g, r = cv2.split(new_image)


def filter_region_mask(region_mask, min_area=100):
    contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) >= min_area]
    
    new_mask = np.zeros_like(region_mask)
    cv2.drawContours(new_mask, large_contours, -1, 255, -1)  # Fill vùng hợp lệ
    return new_mask

# ----------------- REGION 1: Core Glare -----------------
# region1_mask = ((b == 255) & (g == 255) & (r == 255)).astype(np.uint8) * 255
region1_mask = (
    (b >= 250) & (b <= 255) &
    (g >= 250) & (g <= 255) &
    (r >= 250) & (r <= 255)
).astype(np.uint8) * 255

# ----------------- REGION 2: Strong Glare -----------------
cond_b_2 = (b >= 200)
cond_g_2 = (g >= 200)
cond_r_2 = (r >= 200)
sum_2 = cond_b_2.astype(np.uint8) + cond_g_2.astype(np.uint8) + cond_r_2.astype(np.uint8)
region2_mask = ((sum_2 >= 2) & (region1_mask == 0)).astype(np.uint8) * 255

# ----------------- REGION 3: Mild Glare -----------------
cond_b_3 = (b >= 150) & (b < 200)
cond_g_3 = (g >= 150) & (g < 200)
cond_r_3 = (r >= 150) & (r < 200)
sum_3 = cond_b_3.astype(np.uint8) + cond_g_3.astype(np.uint8) + cond_r_3.astype(np.uint8)
region3_mask = ((sum_3 >= 2) & (region1_mask == 0) & (region2_mask == 0)).astype(np.uint8) * 255

# ----------------- TÌM CONTOURS -----------------
contours1, _ = cv2.findContours(region1_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2, _ = cv2.findContours(region2_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours3, _ = cv2.findContours(region3_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# ----------------- VẼ CONTOURS TRÊN ẢNH -----------------
region1_mask = filter_region_mask(region1_mask, min_area=100)
region2_mask = filter_region_mask(region2_mask, min_area=100)
region3_mask = filter_region_mask(region3_mask, min_area=100)
output_contour = new_image.copy()

cv2.drawContours(output_contour, contours1, -1, (0, 0, 255), 1)   # Vùng 1: đỏ
cv2.drawContours(output_contour, contours2, -1, (0, 255, 255), 1) # Vùng 2: vàng
cv2.drawContours(output_contour, contours3, -1, (255, 0, 0), 1)   # Vùng 3: blue
cv.imshow('Original Image', img)
cv.imshow('Anh giam toi', output_contour)
# Wait until user press some key
cv.waitKey()
cv.destroyAllWindows()

 Basic Linear Transforms 
-------------------------


In [47]:
# Ảnh ban đầu, tạo 1 mask lưu vị trí của contour : bên trong = 255, bên ngoài = 0
# Ảnh ban đầu, tăng sáng = ảnh 1
# Ảnh ban đầu, giảm sáng = ảnh 2
# Lấy ảnh 1, loại bỏ vùng mask 255 của ảnh 1
# Lấy ảnh 2, loại bỏ vùng mask 0 của ảnh 2
# Rồi gộp ảnh

import cv2
import numpy as np

img = cv2.imread(r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_60_resize.jpg")
if img is None:
    print("Không thể đọc ảnh. Kiểm tra lại đường dẫn!")
    exit()  
img_copy = img.copy()
b, g, r = cv2.split(img)
mask_b = b > 110
mask_g = g > 110
mask_r = r > 110
mask_sum = mask_b.astype(np.uint8) + mask_g.astype(np.uint8) + mask_r.astype(np.uint8)
final_mask = (mask_sum >= 2).astype(np.uint8) * 255
contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
min_area = 100
large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]
contour_mask = np.zeros(img.shape[:2], dtype=np.uint8)
cv2.drawContours(contour_mask, large_contours, -1, 255, -1) 



img_copy1 = img.copy()
# Tạo ảnh 1
alpha1 = 1
beta1 = 50
anh_1 = cv2.convertScaleAbs(img_copy1, alpha=alpha1, beta=beta1)  
img_copy1[contour_mask == 0] = anh_1[contour_mask == 0]


# Tạo ảnh 2

img_copy2 = img.copy()
alpha2 = 1
beta2 = -50
anh_2 = cv2.convertScaleAbs(img_copy2, alpha=alpha2, beta=beta2)  
img_copy2[contour_mask == 255] = anh_2[contour_mask == 255]

# Tạo ảnh 3
img_copy1[contour_mask == 255] = anh_2[contour_mask == 255]

img1 = img.copy()
img2 = img.copy()
img11 = cv2.convertScaleAbs(img1, alpha=alpha1, beta=beta1)
img22 = cv2.convertScaleAbs(img2, alpha=alpha2, beta=beta2)
img3 = cv2.add(img11 , img22)
img33 = cv2.mean(img3)


cv2.imshow('oi', img_copy)
# cv2.imshow('oidoioi1',img_copy1)
# cv2.imshow('oidoioi2',img_copy2)
cv2.imshow('oidoioi3',img33)
cv2.waitKey()
cv2.destroyAllWindows()


# Resize ảnh

In [ ]:
import cv2
img = cv2.imread(r"D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original\frame_60.jpg")
resized_img = cv2.resize(img, (img.shape[1] // 2, img.shape[0] // 2))
cv2.imwrite(r'D:\SYLLABUS\AIP491\demo\code\glare-reduction\Test_Original/frame_60_resize.jpg', resized_img)

True